#### Etape 3.3 : Detection d'anomalies
- Identifier les pics de consommation anormaux (>3 ecarts-types)
- Detecter les periodes de sous-consommation suspectes (batiment ferme non declare)
- Reperer les batiments dont la consommation ne correspond pas a leur DPE
- Lister les batiments necessitant un audit energetique

In [1]:
import os
import pandas as pd

INPUT_DIR = os.path.join("../output", "consommations_enrichies.csv")

df_conso = pd.read_csv(INPUT_DIR)

print("colonnes :\n",df_conso.dtypes)
print("nb lignes :\n", len(df_conso))

colonnes :
 Unnamed: 0                            int64
batiment_id                             str
type                                    str
commune                                 str
type_energie                            str
conso_clean                         float64
unite                                   str
date                                    str
hour                                  int64
year                                  int64
month                                 int64
classe_energetique                      str
conso_annuelle                      float64
conso_moyenne_par_occupant_annee    float64
conso_m2                            float64
conso_m2_annee                      float64
cout_horaire                        float64
cout_juor                           float64
cout_mois                           float64
cout_annee                          float64
ipe                                 float64
ecart_ipe                           float64
dtype: object
nb lig

In [2]:
conso_moyenne = df_conso["conso_clean"].mean()
ec_type = df_conso["conso_clean"].std()

seuil_haut = conso_moyenne + 3 * ec_type
seuil_bas = conso_moyenne - 3 * ec_type

#### Détection des anomalies

In [13]:
df_anomalies_bas = df_conso[
    (df_conso["conso_clean"] < seuil_bas)
]

df_anomalies_haut = df_conso[
    (df_conso["conso_clean"] > seuil_haut)
]


print("anomalies basses")
print(df_anomalies_bas)


print("anomalies hautes")
print(df_anomalies_haut)

anomalies basses
Empty DataFrame
Columns: [Unnamed: 0, batiment_id, type, commune, type_energie, conso_clean, unite, date, hour, year, month, classe_energetique, conso_annuelle, conso_moyenne_par_occupant_annee, conso_m2, conso_m2_annee, cout_horaire, cout_juor, cout_mois, cout_annee, ipe, ecart_ipe, seuil_bas, sous_conso]
Index: []

[0 rows x 24 columns]
anomalies hautes
         Unnamed: 0 batiment_id     type commune type_energie  conso_clean  \
5206           5206     BAT0005  piscine   Paris  electricite      2711.77   
9458           9458     BAT0005  piscine   Paris          gaz      4668.70   
15429         15429     BAT0005  piscine   Paris  electricite      3085.20   
19158         19158     BAT0005  piscine   Paris          gaz      3865.52   
25664         25664     BAT0005  piscine   Paris  electricite      2782.38   
...             ...         ...      ...     ...          ...          ...   
7460713     7460713     BAT0146  piscine  Toulon          gaz      2015.85   
7

#### sous consommation

In [20]:
seuils_bas = df_conso.groupby("batiment_id")["conso_clean"].quantile(0.1) #on prend les 10% les pls bas

print(len(seuils_bas))

def is_seuil_bas(value):
    return value.quantile(0.1)

df_conso["seuil_bas"] = df_conso \
    .groupby("batiment_id")["conso_clean"] \
    .transform(is_seuil_bas)


df_conso["sous_conso"] = df_conso["conso_clean"] < df_conso["seuil_bas"]

#on sort pour avoir des sous consos contiguës
df_conso = df_conso.sort_values(
    ["batiment_id", "year", "month", "date", "hour"]
)

print(df_conso[~df_conso["sous_conso"]])




146
         Unnamed: 0 batiment_id     type commune type_energie  conso_clean  \
5140           5140     BAT0001    ecole   Paris  electricite         5.70   
9406           9406     BAT0001    ecole   Paris          gaz         7.84   
3413           3413     BAT0001    ecole   Paris  electricite         7.55   
8135           8135     BAT0001    ecole   Paris          gaz         9.58   
6387           6387     BAT0001    ecole   Paris  electricite         6.11   
...             ...         ...      ...     ...          ...          ...   
7484991     7484991     BAT0146  piscine  Toulon          eau       185.12   
7487121     7487121     BAT0146  piscine  Toulon  electricite      1772.28   
7490545     7490545     BAT0146  piscine  Toulon          gaz      1920.75   
7486638     7486638     BAT0146  piscine  Toulon  electricite       176.67   
7491376     7491376     BAT0146  piscine  Toulon          gaz       176.73   

        unite        date  hour  year  ...  conso_m2_annee 